In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00


In [ ]:
# Cell 2: Imports and optional login
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

login()

In [ ]:
!unzip -q gemma3-cord-lora-final.zip -d .

In [ ]:
# Cell 3: Model and adapter paths
BASE_MODEL = "google/gemma-3-1b-it"

# IMPORTANT: use the exact name you saved in training (with underscore)
ADAPTER_PATH = "./gemma3-cord-lora-final"   # note the underscore

# 4-bit config (must match training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# Optional: merge for faster inference
# model = model.merge_and_unload()

print("✅ Model ready for inference!")

Loading tokenizer...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loading LoRA adapter...
✅ Model ready for inference!


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.l

In [ ]:
# Cell 4: Define the same processing function used during training
# (This exactly matches what you did in the training notebook)

def process_sample(sample):
    gt_data = json.loads(sample['ground_truth'])
    gt_parse = gt_data['gt_parse']

    # Build document text from menu items + totals
    doc_parts = []
    if 'menu' in gt_parse:
        menu_items = gt_parse['menu']
        if not isinstance(menu_items, list):
            menu_items = []
        for item in menu_items:
            if isinstance(item, dict):
                nm = item.get('nm', '')
                cnt = item.get('cnt', '')
                price = item.get('price', '')
                if nm:
                    doc_parts.append(f"{nm} {cnt} {price}".strip())
            elif isinstance(item, str):
                doc_parts.append(item)
    if 'sub_total' in gt_parse:
        doc_parts.append(f"Subtotal: {gt_parse['sub_total']}")
    if 'total' in gt_parse:
        doc_parts.append(f"Total: {gt_parse['total']}")

    doc_text = " ".join(doc_parts)

    # Extract target fields (same as training: Total, Subtotal)
    target = {}
    for cat, new_name in [("total", "Total"), ("sub_total", "Subtotal")]:
        if cat in gt_parse:
            target[new_name] = gt_parse[cat]
    return doc_text, target

def format_example(doc, target):
    prompt = f"Extract the following information from the document: Total, Subtotal.\n\nDocument:\n{doc}\n\nAnswer:\n"
    answer = json.dumps(target, ensure_ascii=False)
    return prompt + answer

In [ ]:
# Cell 5: Load only 5 samples from the dataset using streaming (fast, no full download)
from datasets import load_dataset
print("Loading 5 samples from CORD validation set...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="validation", streaming=True)
samples = []
for i, sample in enumerate(dataset):
    if i >= 5:
        break
    samples.append(sample)
print(f"Loaded {len(samples)} samples")

Loading 5 samples from CORD validation set...
Loaded 5 samples


In [ ]:
# Cell 6: Run inference on each sample and compare with ground truth
def extract_info(doc_text):
    prompt = f"Extract the following information from the document: Total, Subtotal.\n\nDocument:\n{doc_text}\n\nAnswer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_raw = full_output.split("Answer:\n")[-1].strip()

    # First, try to extract a proper JSON object
    json_pattern = r'\{[^{}]*\}'
    match = re.search(json_pattern, predicted_raw)
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, dict):
                return parsed
        except:
            pass

    # Fallback: parse key: value pairs from the raw text
    result = {}
    for line in predicted_raw.split('\n'):
        line = line.strip()
        if ':' in line:
            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()
            # Remove trailing backticks, quotes, commas
            value = re.sub(r'[`\'",]+$', '', value).strip()
            if key in ['Total', 'Subtotal']:
                result[key] = value

    if result:
        return result
    else:
        return {"error": "Could not parse", "raw": predicted_raw}

# 4. Load a few samples and test
print("Loading 5 samples from validation set...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="validation", streaming=True)
samples = []
for i, sample in enumerate(dataset):
    if i >= 5:
        break
    samples.append(sample)

for idx, sample in enumerate(samples):
    doc_text, target = process_sample(sample)
    print(f"\n{'='*60}")
    print(f"Sample {idx+1}")
    print(f"Document (first 150 chars): {doc_text[:150]}...")
    print(f"Ground Truth: {json.dumps(target, indent=2)}")
    prediction = extract_info(doc_text)
    print(f"Prediction:   {json.dumps(prediction, indent=2)}")

    # Check if the prediction contains the same keys as ground truth
    if isinstance(prediction, dict) and 'error' not in prediction:
        # Check if all keys in target are present in prediction
        all_keys_present = all(k in prediction for k in target.keys())
        if all_keys_present:
            print("✅ Model extracted all required fields!")
        else:
            print(f"⚠️ Missing keys: {set(target.keys()) - set(prediction.keys())}")
    else:
        print("❌ Model failed to extract structured data")

Loading 5 samples from validation set...

Sample 1
Document (first 150 chars): REAL GANACHE 1 16,500 EGG TART 1 13,000 PIZZA TOAST 1 16,000 Total: {'total_price': '45,500', 'cashprice': '50,000', 'changeprice': '4,500'}...
Ground Truth: {
  "Total": {
    "total_price": "45,500",
    "cashprice": "50,000",
    "changeprice": "4,500"
  }
}
Prediction:   {
  "Total": "45,500",
  "Subtotal": "13,000"
}
✅ Model extracted all required fields!

Sample 2
Document (first 150 chars): Total: {'total_price': '23.000', 'cashprice': '50.000', 'changeprice': '27.000'}...
Ground Truth: {
  "Total": {
    "total_price": "23.000",
    "cashprice": "50.000",
    "changeprice": "27.000"
  }
}
Prediction:   {
  "Total": "23.000",
  "Subtotal": "23.000"
}
✅ Model extracted all required fields!

Sample 3
Document (first 150 chars): Subtotal: {'subtotal_price': '18,181', 'tax_price': '1,818'} Total: {'total_price': '20,000', 'cashprice': '100,000', 'changeprice': '80,000'}...
Ground Truth: {
  "Total": {
   